# 🥇 Gold Layer - Feature Engineering for Churn Prediction

## Propósito
Crear dataset analítico final con features engineered para modelos de churn prediction.

## Arquitectura Medallion - Gold
La capa Gold contiene:
* **Features engineered** específicos para el caso de uso de ML
* **Granularidad por cliente** (1 fila = 1 cliente)
* **Métricas RFM** (Recency, Frequency, Monetary)
* **Variables temporales** y de comportamiento
* **Indicadores de engagement** y riesgo de churn
* **Dataset listo para MLflow y AutoML**

## Features Creados
### 1. **RFM (Recency, Frequency, Monetary)**
   - Recencia de última transacción/interacción
   - Frecuencia de uso de productos/servicios
   - Valor monetario (saldos, retiros)

### 2. **Tenure & Lifecycle**
   - Antigüedad como asociado
   - Permanencia en meses
   - Edad del cliente

### 3. **Behavioral Features**
   - Diversificación de productos
   - Uso de canales digitales vs presenciales
   - Tendencias de saldo

### 4. **Risk Indicators**
   - Ha solicitado retiro de aportes
   - Castigo de cartera
   - Variación negativa de saldo
   - Inactividad transaccional

### 5. **Engagement Score**
   - Score compuesto de engagement basado en múltiples dimensiones

## Tabla Gold Creada
**`workspace.churn_gold.churn_prediction_features`** - Dataset ML-ready con 248K+ clientes

---
**Layer:** Gold (Analytics)  
**Última actualización:** 2026-05-17  
**Uso:** Modelos de Churn Prediction con MLflow/AutoML

In [0]:
%run ./00_config

In [0]:
# ================================================================================
# GOLD LAYER: FEATURE ENGINEERING FOR CHURN PREDICTION
# ================================================================================

from pyspark.sql.functions import (
    ntile, percent_rank, when, coalesce, 
    ceil, floor, round as spark_round,
    datediff, months_between, current_date,
    greatest, least, abs as spark_abs
)

print("\n" + "="*80)
print("🥇 GOLD LAYER: FEATURE ENGINEERING PARA CHURN PREDICTION")
print("="*80)

# Leer tabla consolidada Silver
df_silver = spark.table(f"{SILVER_SCHEMA}.clientes_consolidado")

logger.info(f"Registros leídos de Silver: {df_silver.count():,}")
logger.info(f"Columnas en Silver: {len(df_silver.columns)}")

print(f"\n📊 Iniciando feature engineering sobre {df_silver.count():,} clientes...")

# ==============================================================================
# FEATURE ENGINEERING
# ==============================================================================

df_gold = df_silver

# ------------------------------------------------------------------------------
# 1. RECENCY FEATURES (Recencia - cuán reciente es la actividad)
# ------------------------------------------------------------------------------
print("\n🔄 Creando features de RECENCY...")

# Recencia combinada: la mínima de todas las recencias disponibles
df_gold = df_gold.withColumn(
    "recencia_dias_min",
    least(
        coalesce(col("dias_desde_ultima_tx"), lit(999)),
        coalesce(col("dias_desde_ultima_interaccion"), lit(999))
    )
)

# Recencia máxima (inactividad máxima)
df_gold = df_gold.withColumn(
    "recencia_dias_max",
    greatest(
        coalesce(col("dias_desde_ultima_tx"), lit(0)),
        coalesce(col("dias_desde_ultima_interaccion"), lit(0))
    )
)

# Bins de recencia (segmentación)
df_gold = df_gold.withColumn(
    "recencia_segment",
    when(col("recencia_dias_min") <= 30, "Muy_Activo")
    .when(col("recencia_dias_min") <= 90, "Activo")
    .when(col("recencia_dias_min") <= 180, "En_Riesgo")
    .otherwise("Inactivo")
)

# ------------------------------------------------------------------------------
# 2. FREQUENCY FEATURES (Frecuencia - cuán frecuente es el uso)
# ------------------------------------------------------------------------------
print("🔄 Creando features de FREQUENCY...")

# Frecuencia transaccional mensual
df_gold = df_gold.withColumn(
    "frecuencia_tx_total",
    coalesce(col("numero_transacciones"), lit(0))
)

df_gold = df_gold.withColumn(
    "frecuencia_tx_mensual_avg",
    coalesce(col("frecuencia_tx_mensual"), lit(0.0))
)

# Nivel de actividad transaccional
df_gold = df_gold.withColumn(
    "actividad_transaccional",
    when(col("frecuencia_tx_mensual_avg") == 0, "Sin_Actividad")
    .when(col("frecuencia_tx_mensual_avg") < 1, "Baja")
    .when(col("frecuencia_tx_mensual_avg") < 3, "Media")
    .otherwise("Alta")
)

# ------------------------------------------------------------------------------
# 3. MONETARY FEATURES (Valor monetario)
# ------------------------------------------------------------------------------
print("🔄 Creando features MONETARY (valor)...")

# Saldo total de productos
df_gold = df_gold.withColumn(
    "saldo_total",
    coalesce(col("saldo_total_productos"), lit(0.0))
)

# Valor de retiro (si existe)
df_gold = df_gold.withColumn(
    "retiro_valor_total",
    coalesce(col("retiro_valor"), lit(0.0))
)

# Segmentación de valor (saldo)
df_gold = df_gold.withColumn(
    "segmento_valor",
    when(col("saldo_total") == 0, "Sin_Saldo")
    .when(col("saldo_total") < 10000000, "Bajo")  # < 10M
    .when(col("saldo_total") < 50000000, "Medio") # 10M-50M
    .when(col("saldo_total") < 100000000, "Alto") # 50M-100M
    .otherwise("Premium")  # > 100M
)

# ------------------------------------------------------------------------------
# 4. TENURE FEATURES (Antigüedad y ciclo de vida)
# ------------------------------------------------------------------------------
print("🔄 Creando features de TENURE (antigüedad)...")

# Antigüedad en meses
df_gold = df_gold.withColumn(
    "antiguedad_total_meses",
    coalesce(col("antiguedad_meses"), lit(0))
)

# Antigüedad en años
df_gold = df_gold.withColumn(
    "antiguedad_anos",
    (col("antiguedad_total_meses") / 12.0).cast("double")
)

# Segmento de antigüedad
df_gold = df_gold.withColumn(
    "segmento_antiguedad",
    when(col("antiguedad_total_meses") < 6, "Nuevo")  # < 6 meses
    .when(col("antiguedad_total_meses") < 24, "Reciente")  # 6-24 meses
    .when(col("antiguedad_total_meses") < 60, "Establecido")  # 2-5 años
    .otherwise("Veterano")  # > 5 años
)

# ------------------------------------------------------------------------------
# 5. DIVERSIFICATION FEATURES (Diversificación de productos)
# ------------------------------------------------------------------------------
print("🔄 Creando features de DIVERSIFICACIÓN...")

# Número de productos de ahorro
df_gold = df_gold.withColumn(
    "num_productos",
    coalesce(col("num_productos_ahorro"), lit(0))
)

# Nivel de diversificación
df_gold = df_gold.withColumn(
    "nivel_diversificacion_num",
    coalesce(col("nivel_diversificacion"), lit(0))
)

# Segmento de diversificación
df_gold = df_gold.withColumn(
    "segmento_diversificacion",
    when(col("num_productos") == 0, "Sin_Productos")
    .when(col("num_productos") <= 2, "Baja")
    .when(col("num_productos") <= 5, "Media")
    .otherwise("Alta")
)

# ------------------------------------------------------------------------------
# 6. BEHAVIORAL FEATURES (Comportamiento y canales)
# ------------------------------------------------------------------------------
print("🔄 Creando features de COMPORTAMIENTO...")

# Uso de canales digitales
df_gold = df_gold.withColumn(
    "usa_canal_digital",
    when(
        (col("canal_preferido") == "Digital") | 
        (col("usa_linea") == "Si"),
        1
    ).otherwise(0)
)

# Visita física a agencia
df_gold = df_gold.withColumn(
    "visita_fisica_agencia",
    when(col("visita_agencia") == "Si", 1).otherwise(0)
)

# Multicanal (usa más de un canal)
df_gold = df_gold.withColumn(
    "es_multicanal",
    when(
        (col("visita_fisica_agencia") == 1) & (col("usa_canal_digital") == 1),
        1
    ).otherwise(0)
)

# ------------------------------------------------------------------------------
# 7. RISK INDICATORS (Indicadores de riesgo de churn)
# ------------------------------------------------------------------------------
print("🔄 Creando INDICADORES DE RIESGO...")

# Ha solicitado retiro de aportes (fuerte señal de churn)
df_gold = df_gold.withColumn(
    "ha_solicitado_retiro",
    when(col("retiro_valor").isNotNull(), 1).otherwise(0)
)

# Tiene castigo de cartera
df_gold = df_gold.withColumn(
    "tiene_castigo",
    when(col("tiene_castigo_cartera") == "Si", 1).otherwise(0)
)

# Variación negativa de saldo en últimos 6 meses
df_gold = df_gold.withColumn(
    "variacion_saldo_negativa",
    when(coalesce(col("variacion_saldo_6m"), lit(0)) < -0.1, 1).otherwise(0)  # < -10%
)

# Inactivo transaccional
df_gold = df_gold.withColumn(
    "inactivo_transaccional",
    when(col("frecuencia_tx_total") == 0, 1).otherwise(0)
)

# Sin productos de ahorro
df_gold = df_gold.withColumn(
    "sin_productos_ahorro",
    when(col("num_productos") == 0, 1).otherwise(0)
)

# ------------------------------------------------------------------------------
# 8. ENGAGEMENT SCORE (Score compuesto de engagement)
# ------------------------------------------------------------------------------
print("🔄 Calculando ENGAGEMENT SCORE...")

# Componentes del engagement score (0-100)
# 1. Frecuencia transaccional (0-25 puntos)
df_gold = df_gold.withColumn(
    "eng_frecuencia",
    when(col("frecuencia_tx_mensual_avg") >= 5, lit(25))
    .when(col("frecuencia_tx_mensual_avg") >= 2, lit(15))
    .when(col("frecuencia_tx_mensual_avg") >= 0.5, lit(8))
    .otherwise(lit(0))
)

# 2. Diversificación de productos (0-25 puntos)
df_gold = df_gold.withColumn(
    "eng_diversificacion",
    when(col("num_productos") >= 6, lit(25))
    .when(col("num_productos") >= 4, lit(18))
    .when(col("num_productos") >= 2, lit(10))
    .when(col("num_productos") >= 1, lit(5))
    .otherwise(lit(0))
)

# 3. Recencia de actividad (0-25 puntos)
df_gold = df_gold.withColumn(
    "eng_recencia",
    when(col("recencia_dias_min") <= 7, lit(25))
    .when(col("recencia_dias_min") <= 30, lit(20))
    .when(col("recencia_dias_min") <= 90, lit(12))
    .when(col("recencia_dias_min") <= 180, lit(5))
    .otherwise(lit(0))
)

# 4. Uso de canales (0-25 puntos)
df_gold = df_gold.withColumn(
    "eng_canales",
    when(col("es_multicanal") == 1, lit(25))
    .when(col("usa_canal_digital") == 1, lit(15))
    .when(col("visita_fisica_agencia") == 1, lit(10))
    .otherwise(lit(0))
)

# Engagement score total (0-100)
df_gold = df_gold.withColumn(
    "engagement_score",
    col("eng_frecuencia") + 
    col("eng_diversificacion") + 
    col("eng_recencia") + 
    col("eng_canales")
)

# Segmento de engagement
df_gold = df_gold.withColumn(
    "segmento_engagement",
    when(col("engagement_score") >= 70, "Alto")
    .when(col("engagement_score") >= 40, "Medio")
    .when(col("engagement_score") >= 15, "Bajo")
    .otherwise("Muy_Bajo")
)

# ------------------------------------------------------------------------------
# 9. CHURN RISK SCORE (Score de riesgo de churn)
# ------------------------------------------------------------------------------
print("🔄 Calculando CHURN RISK SCORE...")

# Sumar factores de riesgo (cada factor suma puntos de riesgo)
df_gold = df_gold.withColumn(
    "churn_risk_score_raw",
    (col("ha_solicitado_retiro") * 30) +  # Mayor peso
    (col("variacion_saldo_negativa") * 20) +
    (col("inactivo_transaccional") * 15) +
    (col("tiene_castigo") * 15) +
    (col("sin_productos_ahorro") * 10) +
    # Penalizar por baja recencia (convertir días a puntos)
    when(col("recencia_dias_min") > 180, lit(20))
    .when(col("recencia_dias_min") > 90, lit(10))
    .otherwise(lit(0))
)

# Normalizar a escala 0-100
df_gold = df_gold.withColumn(
    "churn_risk_score",
    least(col("churn_risk_score_raw"), lit(100))
)

# Segmento de riesgo de churn
df_gold = df_gold.withColumn(
    "churn_risk_segment",
    when(col("churn_risk_score") >= 60, "Alto_Riesgo")
    .when(col("churn_risk_score") >= 30, "Riesgo_Moderado")
    .when(col("churn_risk_score") >= 10, "Riesgo_Bajo")
    .otherwise("Sin_Riesgo")
)

print("\n✅ Feature engineering completado")

In [0]:
# ==============================================================================
# SELECCIÓN DE FEATURES FINALES PARA ML
# ==============================================================================

# Import missing col function
from pyspark.sql.functions import col

print("\n" + "="*80)
print("📦 SELECCIONANDO FEATURES FINALES PARA ML")
print("="*80)

# Seleccionar columnas finales para la tabla Gold
df_gold_final = df_gold.select(
    # -------------------------------------------------------------------------
    # IDENTIFICADOR
    # -------------------------------------------------------------------------
    col("nit").alias("cliente_id"),
    
    # -------------------------------------------------------------------------
    # DEMOGRÁFICOS BÁSICOS
    # -------------------------------------------------------------------------
    col("sexo"),
    col("edad"),
    col("nivel_ingresos"),
    
    # -------------------------------------------------------------------------
    # TENURE (Antigüedad)
    # -------------------------------------------------------------------------
    col("antiguedad_total_meses"),
    col("antiguedad_anos"),
    col("permanencia_meses"),
    col("segmento_antiguedad"),
    
    # -------------------------------------------------------------------------
    # RFM - RECENCY
    # -------------------------------------------------------------------------
    col("recencia_dias_min"),
    col("recencia_dias_max"),
    col("recencia_segment"),
    
    # -------------------------------------------------------------------------
    # RFM - FREQUENCY
    # -------------------------------------------------------------------------
    col("frecuencia_tx_total"),
    col("frecuencia_tx_mensual_avg"),
    col("actividad_transaccional"),
    
    # -------------------------------------------------------------------------
    # RFM - MONETARY
    # -------------------------------------------------------------------------
    col("saldo_total"),
    col("retiro_valor_total"),
    col("segmento_valor"),
    col("variacion_saldo_6m"),
    
    # -------------------------------------------------------------------------
    # DIVERSIFICACIÓN DE PRODUCTOS
    # -------------------------------------------------------------------------
    col("num_productos"),
    col("nivel_diversificacion_num"),
    col("segmento_diversificacion"),
    
    # -------------------------------------------------------------------------
    # COMPORTAMIENTO Y CANALES
    # -------------------------------------------------------------------------
    col("usa_canal_digital"),
    col("visita_fisica_agencia"),
    col("es_multicanal"),
    col("canal_preferido"),
    
    # -------------------------------------------------------------------------
    # INDICADORES DE RIESGO (Features binarios)
    # -------------------------------------------------------------------------
    col("ha_solicitado_retiro"),
    col("tiene_castigo"),
    col("variacion_saldo_negativa"),
    col("inactivo_transaccional"),
    col("sin_productos_ahorro"),
    
    # -------------------------------------------------------------------------
    # ENGAGEMENT SCORE
    # -------------------------------------------------------------------------
    col("engagement_score"),
    col("segmento_engagement"),
    
    # -------------------------------------------------------------------------
    # CHURN RISK SCORE (Target Proxy)
    # -------------------------------------------------------------------------
    col("churn_risk_score"),
    col("churn_risk_segment"),
    
    # -------------------------------------------------------------------------
    # FEATURES ADICIONALES DE CONTEXTO
    # -------------------------------------------------------------------------
    col("participacion_social"),
    col("uso_creditos"),
    col("retiro_motivo"),
    col("retiro_tendencia_saldo"),
    
    # -------------------------------------------------------------------------
    # METADATA
    # -------------------------------------------------------------------------
    col("timestamp_ingestion"),
    col("load_date")
)

logger.info(f"Features seleccionados: {len(df_gold_final.columns)}")

print(f"\n📊 Dataset Gold ML-ready:")
print(f"   • Clientes: {df_gold_final.count():,}")
print(f"   • Features: {len(df_gold_final.columns)}")

# ==============================================================================
# ESCRIBIR TABLA GOLD
# ==============================================================================

print("\n" + "="*80)
print("💾 ESCRIBIENDO TABLA GOLD")
print("="*80)

gold_table = GOLD_CHURN_TABLE

df_gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .option("delta.autoOptimize.autoCompact", "true") \
    .saveAsTable(gold_table)

logger.info(f"✓ Tabla Gold creada: {gold_table}")

# Optimizar con ZORDER solo por cliente_id (columna con stats)
if get_execution_params()["run_optimization"]:
    print("\n⚡ Ejecutando OPTIMIZE + ZORDER...")
    # ZORDER solo por cliente_id (churn_risk_score es calculado sin stats)
    optimize_delta_table(gold_table, ["cliente_id"])

log_ingestion_stats(df_gold_final, gold_table, "GOLD_FEATURE_ENGINEERING")

print(f"\n✅ Tabla Gold creada exitosamente: {gold_table}")
print(f"   Dataset listo para MLflow, AutoML y modelos de churn prediction")

In [0]:
# ==============================================================================
# EXPORTAR TABLA GOLD COMPLETA A CSV
# ==============================================================================

import pandas as pd
from datetime import datetime

print("\n" + "="*80)
print("📥 EXPORTANDO TABLA GOLD A CSV")
print("="*80)

# Leer tabla Gold completa
df_export = spark.table("workspace.churn_gold.churn_prediction_features")

print(f"\n📊 Registros a exportar: {df_export.count():,}")
print(f"   Features: {len(df_export.columns)}")

# Convertir a Pandas (para datasets de este tamaño es manejable)
print("\n⏳ Convirtiendo Spark DataFrame a Pandas...")
df_pandas = df_export.toPandas()

print(f"✓ Conversión completada: {len(df_pandas):,} filas")

# Generar nombre de archivo con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"churn_prediction_features_{timestamp}.csv"

# Exportar a CSV (se guardará en el navegador del usuario)
print(f"\n💾 Exportando a CSV: {filename}")

# Crear el CSV y prepararlo para descarga
csv_data = df_pandas.to_csv(index=False)

# Guardar localmente en el driver (para descarga manual)
local_path = f"/tmp/{filename}"
df_pandas.to_csv(local_path, index=False, encoding='utf-8')

print(f"\n✅ Exportación completada!")
print(f"\n📁 Archivo CSV generado:")
print(f"   • Nombre: {filename}")
print(f"   • Filas: {len(df_pandas):,}")
print(f"   • Columnas: {len(df_pandas.columns)}")
print(f"   • Tamaño: {len(csv_data) / (1024*1024):.2f} MB")
print(f"\n💡 El archivo se guardó temporalmente en: {local_path}")
print("\n🔽 Opciones para descargar:")
print("   1. Usa el siguiente código en una celda para ver una muestra:")
print("      df_pandas.head(20)")
print("\n   2. Para descargarlo, usa este comando en otra celda:")
print(f"      dbutils.fs.cp('file:{local_path}', 'dbfs:/FileStore/{filename}')")
print(f"      # Luego descarga desde: https://<tu-workspace>.cloud.databricks.com/files/{filename}")

# Mostrar muestra de los datos
print("\n📊 MUESTRA DE DATOS (primeras 10 filas):")
print("="*80)
display(df_pandas.head(10))

In [0]:
# ==============================================================================
# RESUMEN FINAL - GOLD LAYER
# ==============================================================================

# Import missing col function
from pyspark.sql.functions import col

print("\n" + "="*80)
print("🎯 RESUMEN FINAL - GOLD LAYER")
print("="*80)

# Estadísticas finales
df_gold_final = spark.table(GOLD_CHURN_TABLE)

total_clientes = df_gold_final.count()
total_features = len(df_gold_final.columns)

# Conteos por segmento de riesgo
churn_high_risk = df_gold_final.filter(col("churn_risk_segment") == "Alto_Riesgo").count()
churn_moderate = df_gold_final.filter(col("churn_risk_segment") == "Riesgo_Moderado").count()
churn_low = df_gold_final.filter(col("churn_risk_segment") == "Riesgo_Bajo").count()

# Conteos por engagement
engagement_high = df_gold_final.filter(col("segmento_engagement") == "Alto").count()
engagement_low = df_gold_final.filter(col("segmento_engagement").isin(["Bajo", "Muy_Bajo"])).count()

print(f"""
{'='*80}
TABLA GOLD CREADA
{'='*80}
Tabla:               {GOLD_CHURN_TABLE}
Clientes totales:    {total_clientes:,}
Features totales:    {total_features}

{'='*80}
DISTRIBUCIÓN DE RIESGO DE CHURN
{'='*80}
Alto Riesgo:         {churn_high_risk:,} clientes ({churn_high_risk/total_clientes*100:.1f}%)
Riesgo Moderado:     {churn_moderate:,} clientes ({churn_moderate/total_clientes*100:.1f}%)
Riesgo Bajo:         {churn_low:,} clientes ({churn_low/total_clientes*100:.1f}%)

{'='*80}
DISTRIBUCIÓN DE ENGAGEMENT
{'='*80}
Engagement Alto:     {engagement_high:,} clientes ({engagement_high/total_clientes*100:.1f}%)
Engagement Bajo:     {engagement_low:,} clientes ({engagement_low/total_clientes*100:.1f}%)

{'='*80}
FEATURES CATEGORÍAS
{'='*80}
• Demográficos:       4 features
• Tenure:             4 features
• RFM (Recency):      3 features
• RFM (Frequency):    3 features
• RFM (Monetary):     4 features
• Diversificación:    3 features
• Comportamiento:     4 features
• Risk Indicators:    5 features
• Engagement:         2 features
• Churn Risk:         2 features
• Contexto:           4 features
• Metadata:           2 features

{'='*80}
✅ PIPELINE MEDALLION COMPLETADO
{'='*80}
Bronze → Silver → Gold

🥉 Bronze:  5 tablas (ingesta cruda)
🥈 Silver:  6 tablas (transformaciones + consolidado)
🥇 Gold:    1 tabla (features para ML)

Dataset listo para:
  • MLflow Model Training
  • Databricks AutoML
  • Feature Store
  • Modelos de Churn Prediction

📍 Siguiente paso: Entrenar modelo de churn con MLflow o AutoML
   Comando ejemplo:
   import databricks.automl
   summary = databricks.automl.classify(
       df=spark.table("{GOLD_CHURN_TABLE}"),
       target_col="churn_risk_segment",
       timeout_minutes=30
   )
{'='*80}
""")

In [0]:
%sql
-- Vista SQL: Top 20 clientes con mayor riesgo de churn
SELECT 
    cliente_id,
    churn_risk_score,
    churn_risk_segment,
    engagement_score,
    segmento_engagement,
    ha_solicitado_retiro,
    variacion_saldo_negativa,
    inactivo_transaccional,
    saldo_total,
    antiguedad_anos,
    recencia_dias_min,
    num_productos
FROM workspace.churn_gold.churn_prediction_features
WHERE churn_risk_score >= 50
ORDER BY churn_risk_score DESC
LIMIT 20;

In [0]:
# ==============================================================================
# EXPORTAR A FILESTORE PARA DESCARGA DIRECTA
# ==============================================================================

import pandas as pd

print("\n" + "="*80)
print("📥 EXPORTANDO TABLA GOLD A FILESTORE")
print("="*80)

# Leer tabla Gold completa
df_spark = spark.table("workspace.churn_gold.churn_prediction_features")

total_rows = df_spark.count()
print(f"\n📊 Registros a exportar: {total_rows:,}")
print(f"   Features: {len(df_spark.columns)}")

# Convertir a Pandas
print("\n⏳ Convirtiendo a Pandas DataFrame...")
df_pandas = df_spark.toPandas()
print(f"✓ Convertido: {len(df_pandas):,} filas")

# Guardar CSV temporalmente
local_csv = "/tmp/churn_prediction_features_full.csv"
df_pandas.to_csv(local_csv, index=False, encoding='utf-8')
print(f"✓ CSV temporal creado")

# Copiar a FileStore (descargable vía URL)
filestore_path = "dbfs:/FileStore/exports/churn_prediction_features_full.csv"
print(f"\n⏳ Copiando a FileStore...")

dbutils.fs.mkdirs("dbfs:/FileStore/exports/")
dbutils.fs.cp(f"file:{local_csv}", filestore_path, recurse=True)

print(f"✓ Archivo disponible en FileStore")

# Obtener URL de descarga
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().browserHostName().get()
download_url = f"https://{workspace_url}/files/exports/churn_prediction_features_full.csv"

print("\n" + "="*80)
print("✅ ¡EXPORTACIÓN COMPLETADA!")
print("="*80)
print(f"\n📁 Archivo: churn_prediction_features_full.csv")
print(f"   Filas: {len(df_pandas):,}")
print(f"   Columnas: {len(df_pandas.columns)}")
print(f"   Tamaño: ~{len(df_pandas) * len(df_pandas.columns) * 20 / (1024*1024):.1f} MB")
print(f"\n🔗 DESCARGA DIRECTA:")
print(f"\n   {download_url}")
print(f"\n💡 Copia y pega esa URL en tu navegador para descargar")
print(f"\n   O navega a: Data > DBFS > FileStore > exports")
print("\n" + "="*80)

# Preview
print("\n📋 Preview (10 primeras filas):")
display(df_pandas.head(10))